# Credit Offer Acceptance Prediction

Предсказание принятия кредитного оффера. Метрика — ROC-AUC.

Ключевая особенность данных: train и test разделены по времени, поэтому все проверки выполняются на временных фолдах.

Previous public leaderboard score: **0.759966**.

## 1. Setup

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score

import xgboost as xgb
from catboost import CatBoostClassifier


SEED = 42

TARGET = "target_value"
ID_COL = "front_id"
DATE_COL = "decision_day"

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")

TRAIN_PATH = DATA_DIR / "train_apps.csv"
TEST_PATH = DATA_DIR / "test_apps.csv"

OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 200)

In [ ]:
def detect_xgb_device() -> str:
    x = np.array([[0.0], [1.0], [2.0], [3.0]])
    y = np.array([0, 0, 1, 1])

    try:
        model = xgb.XGBClassifier(
            n_estimators=2,
            tree_method="hist",
            device="cuda",
            random_state=SEED,
        )
        model.fit(x, y, verbose=False)
        return "cuda"
    except Exception:
        return "cpu"


def detect_catboost_task_type() -> str:
    x = pd.DataFrame({"x": [0.0, 1.0, 2.0, 3.0]})
    y = np.array([0, 0, 1, 1])

    try:
        model = CatBoostClassifier(
            iterations=2,
            task_type="GPU",
            devices="0",
            verbose=False,
            allow_writing_files=False,
        )
        model.fit(x, y)
        return "GPU"
    except Exception:
        return "CPU"


XGB_DEVICE = detect_xgb_device()
CAT_TASK_TYPE = detect_catboost_task_type()

print("XGBoost:", XGB_DEVICE)
print("CatBoost:", CAT_TASK_TYPE)

## 2. Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train[DATE_COL] = pd.to_datetime(train[DATE_COL])
test[DATE_COL] = pd.to_datetime(test[DATE_COL])

summary = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train), len(test)],
    "start": [train[DATE_COL].min(), test[DATE_COL].min()],
    "end": [train[DATE_COL].max(), test[DATE_COL].max()],
})

display(summary)
print("Positive rate:", round(train[TARGET].mean(), 4))

In [ ]:
monthly_target = (
    train
    .set_index(DATE_COL)
    .resample("MS")[TARGET]
    .agg(["size", "mean"])
)

plt.figure(figsize=(11, 4))
plt.plot(monthly_target.index, monthly_target["mean"], marker="o")
plt.title("Monthly target rate")
plt.xlabel("Month")
plt.ylabel("Target rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Temporal validation

Используются три expanding-window фолда. Последний период train оставлен отдельным holdout.

In [ ]:
FOLDS = [
    {
        "name": "fold_1",
        "train_end": "2024-10-01",
        "valid_end": "2024-12-01",
    },
    {
        "name": "fold_2",
        "train_end": "2024-12-01",
        "valid_end": "2025-02-01",
    },
    {
        "name": "fold_3",
        "train_end": "2025-02-01",
        "valid_end": "2025-04-01",
    },
]

HOLDOUT_START = pd.Timestamp("2025-04-01")


def get_time_split(
    df: pd.DataFrame,
    train_end: str,
    valid_end: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_end = pd.Timestamp(train_end)
    valid_end = pd.Timestamp(valid_end)

    train_part = df.loc[df[DATE_COL] < train_end].copy()
    valid_part = df.loc[
        (df[DATE_COL] >= train_end)
        & (df[DATE_COL] < valid_end)
    ].copy()

    return train_part, valid_part


fold_table = pd.DataFrame(FOLDS)
display(fold_table)

## 4. Feature engineering

Используются:

- признаки пропусков;
- динамика активности за 30/90 дней;
- rank-based признаки;
- residual-признаки;
- отношения между запросом, лимитом и балансом.

Все статистики fit-ятся только на обучающей части фолда.

In [ ]:
@dataclass
class ResidualSpec:
    x_col: str
    y_col: str
    output_col: str
    edges: np.ndarray
    bin_medians: dict[int, float]
    global_median: float


@dataclass
class FeatureState:
    rank_reference: dict[str, np.ndarray]
    residual_specs: list[ResidualSpec]
    missing_cols: list[str]


RANK_COLUMNS = [
    "cnt_deb_ul_ip_30",
    "cnt_deb_ul_ip_90",
    "sum_deb_ul_30",
    "sum_deb_ul_90",
    "loan_amount_last",
    "overdraft_limit_min",
    "overdraft_limit_max",
    "offered_rate",
    "cb_rate",
    "balance_rur_amt_30_min",
    "cnt_deb_loan_90",
    "cnt_cred_loan_90",
]

RESIDUAL_PAIRS = [
    (
        "cnt_deb_ul_ip_90",
        "cnt_deb_ul_ip_30",
        "cnt_activity_30_residual",
    ),
    (
        "sum_deb_ul_90",
        "sum_deb_ul_30",
        "sum_activity_30_residual",
    ),
    (
        "cb_rate",
        "offered_rate",
        "offered_rate_residual",
    ),
]

In [ ]:
def fit_rank_reference(series: pd.Series) -> np.ndarray:
    return np.sort(series.dropna().to_numpy(dtype=float))


def empirical_rank(
    series: pd.Series,
    reference: np.ndarray,
) -> pd.Series:
    result = pd.Series(np.nan, index=series.index, dtype=float)
    mask = series.notna()

    if len(reference) == 0:
        return result

    values = series.loc[mask].to_numpy(dtype=float)

    result.loc[mask] = (
        np.searchsorted(reference, values, side="right")
        / len(reference)
    )

    return result


def fit_residual_spec(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    output_col: str,
    q: int = 20,
) -> ResidualSpec:
    temp = df[[x_col, y_col]].dropna().copy()

    if temp.empty:
        return ResidualSpec(
            x_col=x_col,
            y_col=y_col,
            output_col=output_col,
            edges=np.array([-np.inf, np.inf]),
            bin_medians={0: 0.0},
            global_median=0.0,
        )

    global_median = float(temp[y_col].median())

    _, edges = pd.qcut(
        temp[x_col],
        q=q,
        retbins=True,
        duplicates="drop",
    )

    edges = np.unique(edges.astype(float))

    if len(edges) < 2:
        edges = np.array([-np.inf, np.inf])
    else:
        edges[0] = -np.inf
        edges[-1] = np.inf

    bins = pd.cut(
        temp[x_col],
        bins=edges,
        labels=False,
        include_lowest=True,
    )

    medians = (
        temp.assign(_bin=bins)
        .groupby("_bin", observed=True)[y_col]
        .median()
        .to_dict()
    )

    return ResidualSpec(
        x_col=x_col,
        y_col=y_col,
        output_col=output_col,
        edges=edges,
        bin_medians={int(k): float(v) for k, v in medians.items()},
        global_median=global_median,
    )


def apply_residual_spec(
    df: pd.DataFrame,
    spec: ResidualSpec,
) -> pd.Series:
    bins = pd.cut(
        df[spec.x_col],
        bins=spec.edges,
        labels=False,
        include_lowest=True,
    )

    expected = (
        bins
        .map(spec.bin_medians)
        .fillna(spec.global_median)
        .astype(float)
    )

    return df[spec.y_col] - expected

In [ ]:
def fit_feature_state(df: pd.DataFrame) -> FeatureState:
    rank_reference = {
        col: fit_rank_reference(df[col])
        for col in RANK_COLUMNS
        if col in df.columns
    }

    residual_specs = [
        fit_residual_spec(df, x_col, y_col, output_col)
        for x_col, y_col, output_col in RESIDUAL_PAIRS
        if x_col in df.columns and y_col in df.columns
    ]

    feature_cols = [
        col for col in df.columns
        if col not in {TARGET, ID_COL, DATE_COL}
    ]

    missing_cols = [
        col for col in feature_cols
        if df[col].isna().any()
    ]

    return FeatureState(
        rank_reference=rank_reference,
        residual_specs=residual_specs,
        missing_cols=missing_cols,
    )


def transform_features(
    df: pd.DataFrame,
    state: FeatureState,
    include_missing_pattern: bool = False,
) -> pd.DataFrame:
    out = df.copy()
    dates = pd.to_datetime(out[DATE_COL])

    # Calendar
    out["year"] = dates.dt.year
    out["month"] = dates.dt.month
    out["quarter"] = dates.dt.quarter
    out["dayofweek"] = dates.dt.dayofweek
    out["dayofmonth"] = dates.dt.day
    out["days_from_2024_02_01"] = (
        dates - pd.Timestamp("2024-02-01")
    ).dt.days

    # Missingness
    missing_cols = [
        col for col in state.missing_cols
        if col in out.columns
    ]

    out["missing_count"] = out[missing_cols].isna().sum(axis=1)

    flags = {
        "has_corp_dashboard_data": "corp_credit_products",
        "has_recent_loan_activity_data": "cnt_deb_loan_90",
        "has_application_history": "app_term_mean_360",
        "has_overdraft_history": "overdraft_app_term_max_360",
        "has_revolving_non_fin_history": "loan_rev_max_start_non_fin",
    }

    for new_col, source_col in flags.items():
        if source_col in out.columns:
            out[new_col] = out[source_col].notna().astype("int8")

    if include_missing_pattern and missing_cols:
        out["missing_pattern"] = (
            out[missing_cols]
            .isna()
            .astype("int8")
            .astype(str)
            .agg("".join, axis=1)
        )

    # 30/90-day dynamics
    if {"cnt_deb_ul_ip_30", "cnt_deb_ul_ip_90"}.issubset(out.columns):
        out["cnt_activity_delta_30_90"] = (
            out["cnt_deb_ul_ip_30"] - out["cnt_deb_ul_ip_90"]
        )

    if {"sum_deb_ul_30", "sum_deb_ul_90"}.issubset(out.columns):
        out["sum_activity_delta_30_90"] = (
            out["sum_deb_ul_30"] - out["sum_deb_ul_90"]
        )

    # Offer / demand gaps
    if {"overdraft_limit_min", "overdraft_limit_max"}.issubset(out.columns):
        out["limit_range"] = (
            out["overdraft_limit_max"] - out["overdraft_limit_min"]
        )

    if {"loan_amount_last", "overdraft_limit_max"}.issubset(out.columns):
        out["loan_vs_limit_max"] = (
            out["loan_amount_last"] - out["overdraft_limit_max"]
        )

    if {"loan_amount_last", "overdraft_limit_min"}.issubset(out.columns):
        out["loan_vs_limit_min"] = (
            out["loan_amount_last"] - out["overdraft_limit_min"]
        )

    # Empirical ranks
    for col, reference in state.rank_reference.items():
        out[f"{col}_rank"] = empirical_rank(out[col], reference)

    rank_diffs = {
        "cnt_activity_rank_delta": (
            "cnt_deb_ul_ip_30_rank",
            "cnt_deb_ul_ip_90_rank",
        ),
        "sum_activity_rank_delta": (
            "sum_deb_ul_30_rank",
            "sum_deb_ul_90_rank",
        ),
        "loan_vs_limit_max_rank": (
            "loan_amount_last_rank",
            "overdraft_limit_max_rank",
        ),
        "rate_rank_gap": (
            "offered_rate_rank",
            "cb_rate_rank",
        ),
        "loan_need_vs_balance_rank": (
            "loan_amount_last_rank",
            "balance_rur_amt_30_min_rank",
        ),
        "limit_vs_balance_rank": (
            "overdraft_limit_max_rank",
            "balance_rur_amt_30_min_rank",
        ),
        "loan_flow_rank_gap": (
            "cnt_deb_loan_90_rank",
            "cnt_cred_loan_90_rank",
        ),
    }

    for new_col, (left_col, right_col) in rank_diffs.items():
        if {left_col, right_col}.issubset(out.columns):
            out[new_col] = out[left_col] - out[right_col]

    # Residuals
    for spec in state.residual_specs:
        out[spec.output_col] = apply_residual_spec(out, spec)

    return out.drop(
        columns=[TARGET, ID_COL, DATE_COL],
        errors="ignore",
    )

### Model matrices

In [ ]:
NO_ABSOLUTE_TIME = [
    "year",
    "days_from_2024_02_01",
]


def prepare_xgb_data(
    train_df: pd.DataFrame,
    other_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    state = fit_feature_state(train_df)

    X_train = transform_features(
        train_df,
        state,
        include_missing_pattern=False,
    )

    X_other = transform_features(
        other_df,
        state,
        include_missing_pattern=False,
    )

    cat_cols = X_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    X_train = pd.get_dummies(
        X_train,
        columns=cat_cols,
        dummy_na=True,
        dtype="int8",
    )

    X_other = pd.get_dummies(
        X_other,
        columns=cat_cols,
        dummy_na=True,
        dtype="int8",
    )

    X_other = X_other.reindex(
        columns=X_train.columns,
        fill_value=0,
    )

    return X_train, X_other


def prepare_cat_data(
    train_df: pd.DataFrame,
    other_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    state = fit_feature_state(train_df)

    X_train = transform_features(
        train_df,
        state,
        include_missing_pattern=True,
    )

    X_other = transform_features(
        other_df,
        state,
        include_missing_pattern=True,
    )

    cat_cols = X_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    for col in cat_cols:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_other[col] = X_other[col].fillna("__MISSING__").astype(str)

    return X_train, X_other, cat_cols

## 5. Models

In [ ]:
XGB_PARAMS = {
    "n_estimators": 500,
    "learning_rate": 0.04,
    "max_depth": 6,
    "min_child_weight": 10,
    "subsample": 0.9,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.2,
    "reg_lambda": 2.0,
    "scale_pos_weight": 1.0,
}


CAT_PARAMS = {
    "iterations": 1500,
    "depth": 7,
    "learning_rate": 0.03,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "bagging_temperature": 1.0,
    "scale_pos_weight": 5.0,
}


def make_xgb_model(seed: int = SEED) -> xgb.XGBClassifier:
    return xgb.XGBClassifier(
        **XGB_PARAMS,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device=XGB_DEVICE,
        max_bin=127,
        n_jobs=-1,
        random_state=seed,
    )


def make_cat_model(
    iterations: int | None = None,
    seed: int = SEED,
) -> CatBoostClassifier:
    params = CAT_PARAMS.copy()

    if iterations is not None:
        params["iterations"] = iterations

    gpu_params = (
        {"task_type": "GPU", "devices": "0"}
        if CAT_TASK_TYPE == "GPU"
        else {}
    )

    return CatBoostClassifier(
        **params,
        **gpu_params,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )


def normalized_rank(values: np.ndarray) -> np.ndarray:
    return (
        pd.Series(values)
        .rank(method="average", pct=True)
        .to_numpy()
    )

## 6. Rolling OOF validation

In [ ]:
oof_rows = []
cat_best_iterations = []

for fold in FOLDS:
    print(fold["name"])

    train_part, valid_part = get_time_split(
        train,
        fold["train_end"],
        fold["valid_end"],
    )

    # XGBoost
    X_xgb_train, X_xgb_valid = prepare_xgb_data(
        train_part,
        valid_part,
    )

    xgb_full = make_xgb_model()
    xgb_full.fit(
        X_xgb_train,
        train_part[TARGET],
        verbose=False,
    )

    xgb_full_pred = xgb_full.predict_proba(
        X_xgb_valid
    )[:, 1]

    X_xgb_train_no_time = X_xgb_train.drop(
        columns=NO_ABSOLUTE_TIME,
        errors="ignore",
    )

    X_xgb_valid_no_time = X_xgb_valid.drop(
        columns=NO_ABSOLUTE_TIME,
        errors="ignore",
    )

    xgb_no_time = make_xgb_model()
    xgb_no_time.fit(
        X_xgb_train_no_time,
        train_part[TARGET],
        verbose=False,
    )

    xgb_no_time_pred = xgb_no_time.predict_proba(
        X_xgb_valid_no_time
    )[:, 1]

    # CatBoost
    X_cat_train, X_cat_valid, cat_cols = prepare_cat_data(
        train_part,
        valid_part,
    )

    cat_model = make_cat_model()
    cat_model.fit(
        X_cat_train,
        train_part[TARGET],
        cat_features=cat_cols,
        eval_set=(X_cat_valid, valid_part[TARGET]),
        early_stopping_rounds=100,
        verbose=False,
    )

    cat_pred = cat_model.predict_proba(
        X_cat_valid
    )[:, 1]

    best_iteration = cat_model.get_best_iteration()

    if best_iteration >= 0:
        cat_best_iterations.append(best_iteration + 1)

    oof_rows.append(pd.DataFrame({
        "fold": fold["name"],
        "target": valid_part[TARGET].to_numpy(),
        "xgb_full_pred": xgb_full_pred,
        "xgb_no_time_pred": xgb_no_time_pred,
        "cat_pred": cat_pred,
        "xgb_full_rank": normalized_rank(xgb_full_pred),
        "xgb_no_time_rank": normalized_rank(xgb_no_time_pred),
        "cat_rank": normalized_rank(cat_pred),
    }))


oof = pd.concat(oof_rows, ignore_index=True)

model_scores = []

for fold_name, fold_df in oof.groupby("fold"):
    model_scores.extend([
        {
            "fold": fold_name,
            "model": "XGBoost",
            "auc": roc_auc_score(
                fold_df["target"],
                fold_df["xgb_full_pred"],
            ),
        },
        {
            "fold": fold_name,
            "model": "XGBoost (no absolute time)",
            "auc": roc_auc_score(
                fold_df["target"],
                fold_df["xgb_no_time_pred"],
            ),
        },
        {
            "fold": fold_name,
            "model": "CatBoost",
            "auc": roc_auc_score(
                fold_df["target"],
                fold_df["cat_pred"],
            ),
        },
    ])

model_scores = pd.DataFrame(model_scores)

display(
    model_scores.pivot(
        index="model",
        columns="fold",
        values="auc",
    )
)

## 7. Blend selection

Сначала смешиваются XGBoost и CatBoost. Затем к этому ансамблю добавляется XGBoost без абсолютной временной координаты.

In [ ]:
anchor_rows = []

for xgb_weight in np.linspace(0.0, 1.0, 21):
    fold_scores = {}

    for fold_name, fold_df in oof.groupby("fold"):
        score = (
            xgb_weight * fold_df["xgb_full_rank"]
            + (1.0 - xgb_weight) * fold_df["cat_rank"]
        )

        fold_scores[fold_name] = roc_auc_score(
            fold_df["target"],
            score,
        )

    anchor_rows.append({
        "xgb_weight": float(xgb_weight),
        "cat_weight": float(1.0 - xgb_weight),
        **fold_scores,
        "mean_auc": float(np.mean(list(fold_scores.values()))),
    })


anchor_search = (
    pd.DataFrame(anchor_rows)
    .sort_values("mean_auc", ascending=False)
    .reset_index(drop=True)
)

display(anchor_search.head(10))

BEST_XGB_WEIGHT = float(anchor_search.iloc[0]["xgb_weight"])
BEST_CAT_WEIGHT = 1.0 - BEST_XGB_WEIGHT

print(
    f"Anchor: XGB={BEST_XGB_WEIGHT:.2f}, "
    f"CAT={BEST_CAT_WEIGHT:.2f}"
)

In [ ]:
oof["anchor_rank"] = (
    BEST_XGB_WEIGHT * oof["xgb_full_rank"]
    + BEST_CAT_WEIGHT * oof["cat_rank"]
)


FOLD_WEIGHTS = {
    "fold_1": 0.20,
    "fold_2": 0.30,
    "fold_3": 0.50,
}

final_blend_rows = []

for anchor_weight in np.linspace(0.0, 1.0, 21):
    fold_scores = {}

    for fold_name, fold_df in oof.groupby("fold"):
        score = (
            anchor_weight * fold_df["anchor_rank"]
            + (1.0 - anchor_weight)
            * fold_df["xgb_no_time_rank"]
        )

        fold_scores[fold_name] = roc_auc_score(
            fold_df["target"],
            score,
        )

    weighted_auc = sum(
        FOLD_WEIGHTS[fold] * fold_scores[fold]
        for fold in FOLD_WEIGHTS
    )

    final_blend_rows.append({
        "anchor_weight": float(anchor_weight),
        "no_time_weight": float(1.0 - anchor_weight),
        **fold_scores,
        "recent_weighted_auc": weighted_auc,
    })


final_blend_search = (
    pd.DataFrame(final_blend_rows)
    .sort_values("recent_weighted_auc", ascending=False)
    .reset_index(drop=True)
)

display(final_blend_search.head(10))

BEST_ANCHOR_WEIGHT = float(
    final_blend_search.iloc[0]["anchor_weight"]
)

BEST_NO_TIME_WEIGHT = 1.0 - BEST_ANCHOR_WEIGHT

print(
    f"Final blend: anchor={BEST_ANCHOR_WEIGHT:.2f}, "
    f"no_time={BEST_NO_TIME_WEIGHT:.2f}"
)

## 8. Late holdout

In [ ]:
holdout_train = train.loc[
    train[DATE_COL] < HOLDOUT_START
].copy()

holdout_valid = train.loc[
    train[DATE_COL] >= HOLDOUT_START
].copy()


# XGBoost
X_xgb_train, X_xgb_valid = prepare_xgb_data(
    holdout_train,
    holdout_valid,
)

xgb_full = make_xgb_model()
xgb_full.fit(
    X_xgb_train,
    holdout_train[TARGET],
    verbose=False,
)

holdout_xgb_pred = xgb_full.predict_proba(
    X_xgb_valid
)[:, 1]


X_xgb_train_no_time = X_xgb_train.drop(
    columns=NO_ABSOLUTE_TIME,
    errors="ignore",
)

X_xgb_valid_no_time = X_xgb_valid.drop(
    columns=NO_ABSOLUTE_TIME,
    errors="ignore",
)

xgb_no_time = make_xgb_model()
xgb_no_time.fit(
    X_xgb_train_no_time,
    holdout_train[TARGET],
    verbose=False,
)

holdout_no_time_pred = xgb_no_time.predict_proba(
    X_xgb_valid_no_time
)[:, 1]


# CatBoost
X_cat_train, X_cat_valid, cat_cols = prepare_cat_data(
    holdout_train,
    holdout_valid,
)

cat_model = make_cat_model()
cat_model.fit(
    X_cat_train,
    holdout_train[TARGET],
    cat_features=cat_cols,
    eval_set=(X_cat_valid, holdout_valid[TARGET]),
    early_stopping_rounds=100,
    verbose=False,
)

holdout_cat_pred = cat_model.predict_proba(
    X_cat_valid
)[:, 1]


holdout_anchor = (
    BEST_XGB_WEIGHT * normalized_rank(holdout_xgb_pred)
    + BEST_CAT_WEIGHT * normalized_rank(holdout_cat_pred)
)

holdout_final = (
    BEST_ANCHOR_WEIGHT * holdout_anchor
    + BEST_NO_TIME_WEIGHT
    * normalized_rank(holdout_no_time_pred)
)


holdout_results = pd.DataFrame({
    "model": [
        "XGBoost",
        "CatBoost",
        "XGBoost + CatBoost",
        "XGBoost (no absolute time)",
        "Final blend",
    ],
    "holdout_auc": [
        roc_auc_score(
            holdout_valid[TARGET],
            holdout_xgb_pred,
        ),
        roc_auc_score(
            holdout_valid[TARGET],
            holdout_cat_pred,
        ),
        roc_auc_score(
            holdout_valid[TARGET],
            holdout_anchor,
        ),
        roc_auc_score(
            holdout_valid[TARGET],
            holdout_no_time_pred,
        ),
        roc_auc_score(
            holdout_valid[TARGET],
            holdout_final,
        ),
    ],
}).sort_values(
    "holdout_auc",
    ascending=False,
)

display(holdout_results)

## 9. Final training

In [ ]:
FINAL_CAT_ITERATIONS = (
    int(np.median(cat_best_iterations))
    if cat_best_iterations
    else CAT_PARAMS["iterations"]
)

print("Final CatBoost iterations:", FINAL_CAT_ITERATIONS)

In [ ]:
test_ids = test[ID_COL].copy()


# XGBoost matrices
X_xgb_full, X_xgb_test = prepare_xgb_data(
    train,
    test,
)

X_xgb_full_no_time = X_xgb_full.drop(
    columns=NO_ABSOLUTE_TIME,
    errors="ignore",
)

X_xgb_test_no_time = X_xgb_test.drop(
    columns=NO_ABSOLUTE_TIME,
    errors="ignore",
)


# Full XGBoost
final_xgb = make_xgb_model()
final_xgb.fit(
    X_xgb_full,
    train[TARGET],
    verbose=False,
)

test_xgb_pred = final_xgb.predict_proba(
    X_xgb_test
)[:, 1]


# XGBoost without absolute time
final_xgb_no_time = make_xgb_model()
final_xgb_no_time.fit(
    X_xgb_full_no_time,
    train[TARGET],
    verbose=False,
)

test_no_time_pred = final_xgb_no_time.predict_proba(
    X_xgb_test_no_time
)[:, 1]


# CatBoost
X_cat_full, X_cat_test, cat_cols = prepare_cat_data(
    train,
    test,
)

final_cat = make_cat_model(
    iterations=FINAL_CAT_ITERATIONS,
)

final_cat.fit(
    X_cat_full,
    train[TARGET],
    cat_features=cat_cols,
    verbose=False,
)

test_cat_pred = final_cat.predict_proba(
    X_cat_test
)[:, 1]

## 10. Submission

In [ ]:
test_anchor = (
    BEST_XGB_WEIGHT * normalized_rank(test_xgb_pred)
    + BEST_CAT_WEIGHT * normalized_rank(test_cat_pred)
)

test_score = (
    BEST_ANCHOR_WEIGHT * test_anchor
    + BEST_NO_TIME_WEIGHT
    * normalized_rank(test_no_time_pred)
)


submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: test_score,
})

SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

submission.to_csv(
    SUBMISSION_PATH,
    index=False,
)

display(submission.head())
print("Saved:", SUBMISSION_PATH)

## Notes

- Random split не использовался из-за временного разделения train/test.
- Class weighting ухудшал XGBoost, поэтому оставлен `scale_pos_weight=1`.
- Rank blending использован из-за ROC-AUC и разных шкал вероятностей моделей.
- Абсолютные временные признаки вынесены в отдельную модель, чтобы снизить риск плохой экстраполяции.